In [0]:
# =============================================================================
# ANÁLISE FINAL - Respondendo as perguntas de negócio
# =============================================================================
# Objetivo: Utilizar a tabela Gold para responder as 6 perguntas definidas
# no objetivo do MVP sobre padrão de compra dos clientes da Olist.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import when

# Leitura da tabela Gold
fato = spark.table("workspace.default.gold_fato_pedidos")

# Filtra apenas pedidos entregues para garantir análises sobre compras concluídas
fato_entregue = fato.filter(F.col("order_status") == "delivered")

print(f"Total de itens entregues para análise: {fato_entregue.count():,}")

# =============================================================================
# PERGUNTA 1: Quais categorias concentram maior volume de pedidos e receita?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 1: Volume de pedidos e receita por categoria")
print("="*60)

p1 = (fato_entregue
    .filter(F.col("categoria_en").isNotNull())
    .groupBy("categoria_en")
    .agg(
        F.count("order_id").alias("volume_itens"),
        F.round(F.sum("preco_item"), 2).alias("receita_total"),
        F.round(F.avg("preco_item"), 2).alias("ticket_medio")
    )
    .orderBy("volume_itens", ascending=False)
)

print("\nTop 10 categorias por volume:")
p1.show(10, truncate=False)

# =============================================================================
# PERGUNTA 2: Qual o ticket médio por categoria e como varia entre regiões?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 2: Ticket médio por região")
print("="*60)

fato_regiao = fato_entregue.withColumn("regiao",
    when(F.col("estado_cliente").isin("SP","RJ","ES","MG"), "Sudeste")
    .when(F.col("estado_cliente").isin("PR","SC","RS"), "Sul")
    .when(F.col("estado_cliente").isin("BA","SE","AL","PE","PB","RN","CE","PI","MA"), "Nordeste")
    .when(F.col("estado_cliente").isin("GO","MT","MS","DF"), "Centro-Oeste")
    .when(F.col("estado_cliente").isin("AM","PA","RO","RR","AC","AP","TO"), "Norte")
    .otherwise("Outros")
)

p2 = (fato_regiao
    .filter(F.col("categoria_en").isNotNull())
    .groupBy("regiao")
    .agg(
        F.round(F.avg("preco_item"), 2).alias("ticket_medio"),
        F.count("order_id").alias("volume_itens"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("ticket_medio", ascending=False)
)

print("\nTicket médio por região:")
p2.show(truncate=False)

# =============================================================================
# PERGUNTA 3: Quais formas de pagamento são mais usadas e qual o parcelamento médio?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 3: Formas de pagamento e parcelamento")
print("="*60)

p3 = (fato_entregue
    .filter(F.col("forma_pagamento") != "not_defined")
    .groupBy("forma_pagamento")
    .agg(
        F.count("order_id").alias("volume_pedidos"),
        F.round(F.avg("parcelas"), 1).alias("media_parcelas"),
        F.round(F.avg("valor_total_pedido"), 2).alias("ticket_medio_pedido")
    )
    .orderBy("volume_pedidos", ascending=False)
)

print("\nFormas de pagamento:")
p3.show(truncate=False)

# =============================================================================
# PERGUNTA 4: Como o volume de pedidos evoluiu ao longo dos meses?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 4: Evolução mensal do volume de pedidos")
print("="*60)

p4 = (fato_entregue
    .filter(F.col("ano_compra").isin(2017, 2018))
    .groupBy("ano_compra", "mes_compra")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("ano_compra", "mes_compra")
)

print("\nEvolução mensal (2017-2018):")
p4.show(24, truncate=False)

# =============================================================================
# PERGUNTA 5: Quais estados têm maior concentração de clientes compradores?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 5: Concentração de clientes por estado")
print("="*60)

p5 = (fato_entregue
    .groupBy("estado_cliente")
    .agg(
        F.countDistinct("customer_id").alias("total_clientes"),
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("total_clientes", ascending=False)
)

print("\nTop 10 estados por clientes:")
p5.show(10, truncate=False)

# =============================================================================
# PERGUNTA 6: Existe relação entre avaliação do pedido e valor gasto?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 6: Relação entre avaliação e valor gasto")
print("="*60)

reviews = spark.table("workspace.default.bronze_order_reviews")

# try_cast tolera valores malformados na coluna review_score, convertendo-os para null
reviews_clean = (reviews
    .select("order_id", F.expr("try_cast(review_score as int)").alias("review_score"))
    .filter(F.col("review_score").isNotNull())
    .filter(F.col("review_score").between(1, 5))
)

valor_pedido = (fato_entregue
    .groupBy("order_id")
    .agg(F.round(F.sum("preco_item"), 2).alias("valor_pedido"))
)

p6 = (reviews_clean
    .join(valor_pedido, on="order_id", how="inner")
    .groupBy("review_score")
    .agg(
        F.count("order_id").alias("total_pedidos"),
        F.round(F.avg("valor_pedido"), 2).alias("valor_medio_pedido"),
        F.round(F.min("valor_pedido"), 2).alias("valor_minimo"),
        F.round(F.max("valor_pedido"), 2).alias("valor_maximo")
    )
    .orderBy("review_score")
)

print("\nValor médio gasto por nota de avaliação:")
p6.show(truncate=False)

In [0]:
# =============================================================================
# DISCUSSÃO DOS RESULTADOS
# =============================================================================

print("""
DISCUSSÃO DOS RESULTADOS
========================

PERGUNTA 1: Categorias por volume e receita
-------------------------------------------
Cama, mesa e banho (bed_bath_table) lidera em volume com 10.953 itens,
mas saúde e beleza (health_beauty) gera mais receita (R$ 1,23 mi) com
ticket médio de R$ 130. A categoria relógios e presentes (watches_gifts)
chama atenção: apenas 7ª em volume mas 2ª em receita total (R$ 1,17 mi)
com o maior ticket médio entre as top 10 (R$ 199). Isso indica que
categorias de menor volume podem ser mais estratégicas do ponto de vista
de receita por item vendido.

PERGUNTA 2: Ticket médio por região
------------------------------------
As regiões Norte e Nordeste apresentam ticket médio mais alto (R$ 159 e
R$ 148) do que o Sudeste (R$ 114), apesar de concentrarem bem menos
volume. Uma hipótese é que consumidores nessas regiões compram menos
vezes, mas optam por itens de maior valor quando compram. O Sudeste,
apesar do menor ticket médio, concentra 74.682 itens e R$ 8,5 mi em
receita, representando a maior fatia absoluta do mercado.

PERGUNTA 3: Formas de pagamento
--------------------------------
O cartão de crédito domina com 83.253 transações (75% do total) e
parcelamento médio de 3,6 vezes, confirmando o comportamento típico
do consumidor brasileiro de fracionar compras. O boleto aparece em
segundo com 22.362 transações e ticket médio próximo ao cartão
(R$ 176 vs R$ 182), sugerindo que não é apenas usado para compras
de baixo valor. Vouchers têm o menor ticket médio (R$ 129), indicando
uso predominante em compras menores ou com desconto.

PERGUNTA 4: Evolução mensal
----------------------------
O crescimento é consistente ao longo de 2017, saindo de 750 pedidos
em janeiro para 7.289 em novembro (pico do ano, provavelmente Black
Friday). Em 2018 o patamar se estabiliza entre 6.000 e 7.000 pedidos
mensais, indicando maturidade da operação. O salto entre dezembro/2017
e janeiro/2018 (de 5.513 para 7.069) sugere que o crescimento foi
estrutural e não apenas sazonal.

PERGUNTA 5: Concentração geográfica
-------------------------------------
São Paulo concentra 40.501 clientes, mais do que RJ (12.350) e MG
(11.354) somados. Os três estados do Sudeste juntos representam cerca
de 58% da base de clientes, refletindo a concentração econômica
brasileira. Estados do Norte e Centro-Oeste aparecem pouco no ranking,
o que, combinado com o ticket médio mais alto dessas regiões
(Pergunta 2), reforça a hipótese de compra menos frequente porém
de maior valor nessas localidades.

PERGUNTA 6: Avaliação e valor gasto
-------------------------------------
Pedidos com nota 1 têm o maior valor médio (R$ 164,90), enquanto
notas 2, 3, 4 e 5 ficam entre R$ 127 e R$ 143. Isso sugere uma
relação inversa entre satisfação e valor: compras de maior valor
tendem a gerar mais insatisfação, possivelmente por expectativas
mais altas ou por problemas logísticos em itens maiores. Pedidos
com nota 5 têm o maior volume (57.066), indicando que a maioria
das experiências é positiva independente do valor gasto.

CONCLUSÃO GERAL
---------------
O pipeline revelou um marketplace em forte crescimento no período
analisado, dominado pelo Sudeste em volume mas com regiões periféricas
apresentando maior valor por compra. O cartão de crédito parcelado é
o meio de pagamento predominante e categorias como relógios/presentes
e saúde/beleza se destacam em receita mesmo sem liderar em volume.
A relação inversa entre valor gasto e satisfação aponta para uma
oportunidade de melhoria na experiência pós-venda em compras de
maior ticket.
""")